In [1]:
import os
import shutil
import sqlite3
from datetime import datetime
from PIL import Image

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import resnet34

In [2]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device for inference: {device}")

# Preprocessing transforms (matches ResNet requirements)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Class definitions
class_names = ['star', 'galaxy', 'quasar', 'nebula', 'planet']

Using device for inference: cuda


In [3]:
# Automatically locate the root of the project NASA_TRAINING
current_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.abspath(os.path.join(current_notebook_dir, '..'))

# Locate the .pth file inside training/
weights_path = os.path.join(project_root, 'training', 'trained_net.pth')

# Initialize ResNet34
net = resnet34(weights=None)
num_ftrs = net.fc.in_features
net.fc = nn.Linear(num_ftrs, len(class_names))

if os.path.exists(weights_path):
    net.load_state_dict(torch.load(weights_path, map_location=device))
    net = net.to(device)
    net.eval()
    print(f"Successfully loaded model weights from:\n   {weights_path}")
else:
    raise FileNotFoundError(f"Model file not found at: {weights_path}")

Successfully loaded model weights from:
   /home/rsalas/Documentos/NASA_TRAINING/training/trained_net.pth


In [4]:
# Destination directory: network_sorting/
sorting_dir = os.path.join(project_root, 'network_sorting')

for class_name in class_names:
    os.makedirs(os.path.join(sorting_dir, class_name), exist_ok=True)

# SQLite database setup inside network_sorting/
db_path = os.path.join(sorting_dir, 'classified_images.db')
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS classifications (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        filename TEXT,
        source_directory TEXT,
        predicted_class TEXT,
        destination_path TEXT,
        timestamp TEXT
    )
''')
conn.commit()

print(f"SQLite Database ready: {db_path}")
print(f"Destination folders initialized under network_sorting/")

SQLite Database ready: /home/rsalas/Documentos/NASA_TRAINING/network_sorting/classified_images.db
Destination folders initialized under network_sorting/


In [5]:
def predict_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image_tensor = test_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = net(image_tensor)
        _, predicted = torch.max(output, 1)
    return class_names[predicted.item()]

# Source folders inside images/
source_dirs = [
    os.path.join(project_root, 'images', 'mastDownloads'),
    os.path.join(project_root, 'images', 'nasaDownloads')
]

total_processed = 0
print("STARTING IMAGE CLASSIFICATION & SORTING IN VS CODE\n")

for source_dir in source_dirs:
    if not os.path.exists(source_dir):
        print(f"Source directory not found: {source_dir}")
        continue

    print(f"Scanning folder: {source_dir}...")

    for root, _, files in os.walk(source_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                src_full_path = os.path.join(root, file)

                try:
                    # Run model prediction
                    predicted_class = predict_image(src_full_path)

                    # Destination folder (network_sorting/<predicted_class>/)
                    dst_folder = os.path.join(sorting_dir, predicted_class)
                    dst_full_path = os.path.join(dst_folder, file)

                    # Copy image to classified folder
                    shutil.copy(src_full_path, dst_full_path)

                    # Insert result into SQLite database
                    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    cursor.execute('''
                        INSERT INTO classifications (filename, source_directory, predicted_class, destination_path, timestamp)
                        VALUES (?, ?, ?, ?, ?)
                    ''', (file, root, predicted_class, dst_full_path, current_time))

                    total_processed += 1
                    print(f"[{total_processed:03d}] {file:<25} -> {predicted_class:<8} (Copied & Saved)")

                except Exception as e:
                    print(f"Error processing file {file}: {e}")

# Commit database changes and close connection
conn.commit()
conn.close()

print(f"\nDone! Processed and sorted {total_processed} images into 'network_sorting/'.")
print(f"Results logged to '{db_path}'.")

STARTING IMAGE CLASSIFICATION & SORTING IN VS CODE

Scanning folder: /home/rsalas/Documentos/NASA_TRAINING/images/mastDownloads...
[001] if7p62ofq_flc.jpg         -> star     (Copied & Saved)
[002] if7p62ofq_flt.jpg         -> star     (Copied & Saved)
[003] if7p62ofq_raw.jpg         -> star     (Copied & Saved)
[004] hst_17711_01_acs_sbc_f150lp_jffi01mu_drz.jpg -> star     (Copied & Saved)
[005] ifpv13koq_flc.jpg         -> star     (Copied & Saved)
[006] ifpv13koq_flt.jpg         -> star     (Copied & Saved)
[007] ifpv13koq_raw.jpg         -> star     (Copied & Saved)
[008] hst_17740_06_wfc3_ir_total_ifh506_drz.jpg -> star     (Copied & Saved)
[009] jffi01mqq_raw.jpg         -> star     (Copied & Saved)
[010] jffi01mqq_flt.jpg         -> star     (Copied & Saved)
[011] hst_17535_62_wfc3_uvis_f225w_if7p62oi_drc.jpg -> star     (Copied & Saved)
[012] ifh506n9q_flt.jpg         -> star     (Copied & Saved)
[013] ifh506n9q_raw.jpg         -> planet   (Copied & Saved)
[014] hst_17711_01_ac